# Groq + TWZRD Agent Intel MCP: Web3 Agent Trust Verification

This notebook shows Python developers how to connect Groq with the TWZRD Agent Intel MCP server
to verify Web3 AI agent trustworthiness before making x402 payments.

[TWZRD Agent Intel](https://intel.twzrd.xyz) is a live production MCP server (streamable-http) that
scores any Solana wallet address on a 0–100 trust scale using on-chain signals.

We will do this in three simple steps:
1. Set up the **Groq client** for fast AI responses.
2. Set up the **TWZRD MCP server** for on-chain trust scoring.
3. **Connect them together** using the Responses API.

---


## Getting Started

Follow these steps to set up:
1. **Sign up** for Groq at [console.groq.com](https://console.groq.com/keys) to get your free API key.
2. **TWZRD requires no API key** — the `score_agent` and `preflight_check` tools are free.
3. **Paste your Groq API key** into the cell below and run the cell.


In [ ]:
# To export your API keys into a .env file, run the following cell (replace with your actual key):
!echo "GROQ_API_KEY=<your-groq-api-key>" >> .env

In [ ]:
import json
import os
import time

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY environment variable not set")

print("GROQ_API_KEY:", GROQ_API_KEY[:8] + "...")

## Step 1: Set up the Groq client


In [ ]:
# Model configuration
MODEL = "openai/gpt-oss-120b"

In [ ]:
from openai import OpenAI

# Set up Groq client using the OpenAI-compatible endpoint
client = OpenAI(base_url="https://api.groq.com/api/openai/v1", api_key=GROQ_API_KEY)

## Step 2: Set up TWZRD Agent Intel MCP server

TWZRD Agent Intel exposes three tools via streamable-http at `https://intel.twzrd.xyz/mcp`:

| Tool | Cost | Description |
|------|------|-------------|
| `score_agent` | Free | 0–100 trust score + on-chain risk signals |
| `preflight_check` | Free | Quick go/no-go before x402 payment |
| `get_trust_receipt` | $0.01 USDC (x402) | Signed on-chain trust receipt |

No API key required for the free tools.


In [ ]:
# Set up TWZRD Agent Intel MCP server
# No API key required — free tools are public
tools = [
    {
        "type": "mcp",
        "server_url": "https://intel.twzrd.xyz/mcp",
        "server_label": "twzrd",
        "require_approval": "never",
    }
]

## Step 3: Connect Groq to TWZRD MCP through Groq's Responses API


In [ ]:
def check_agent_trust(client, tools, wallet: str, question: str):
    """Connect Groq to TWZRD for Web3 agent trust verification."""

    start_time = time.time()

    response = client.responses.create(
        model=MODEL,
        input=question,
        tools=tools,
        stream=False,
        temperature=0.1,
        top_p=0.4,
    )

    total_time = time.time() - start_time

    # Extract output text
    output_text = ""
    mcp_calls = {"mcp_calls_performed": []}

    for item in response.output:
        if hasattr(item, "type"):
            if item.type == "message":
                for content in item.content:
                    if hasattr(content, "text"):
                        output_text = content.text
            elif item.type == "mcp_call":
                mcp_calls["mcp_calls_performed"].append(
                    {
                        "type": item.type,
                        "name": getattr(item, "name", "unknown"),
                        "server_label": getattr(item, "server_label", "twzrd"),
                        "arguments": getattr(item, "arguments", "{}"),
                        "output": getattr(item, "output", ""),
                    }
                )

    print(f"\nTime taken: {total_time:.2f} seconds")
    print(f"\nModel response:\n{output_text}")

    return {
        "output": output_text,
        "mcp_calls": mcp_calls,
        "time": total_time,
    }

## Demo 1: Trust score for a known Web3 agent

We check the trust score of `D1QkbFJKiPsymJ65RKHhF6DFB8sPMfpBaFBzuHKfJGWi` — a known
repeat x402 payer from the Dexter ecosystem with 48+ paid transactions.


In [ ]:
WALLET = "D1QkbFJKiPsymJ65RKHhF6DFB8sPMfpBaFBzuHKfJGWi"

trust_check = check_agent_trust(
    client,
    tools,
    WALLET,
    f"Check the trust score for Solana wallet {WALLET} and tell me whether it's safe to make an x402 payment to this agent. Use both score_agent and preflight_check.",
)

## Demo 2: Try it yourself

Replace the wallet address with any Solana wallet you want to check:


In [ ]:
# Replace with any Solana wallet address
MY_WALLET = "<your-solana-wallet-address>"

custom_check = check_agent_trust(
    client,
    tools,
    MY_WALLET,
    f"What is the trust score for Solana wallet {MY_WALLET}? Is it safe for an autonomous agent to pay this wallet via x402?",
)

## Summary

You've just connected Groq's fast inference to TWZRD Agent Intel's on-chain trust scoring.

| What you used | Why |
|--------------|-----|
| **Groq Responses API** | Fast inference with built-in MCP tool execution |
| **TWZRD MCP server** | Live on-chain trust data for any Solana wallet |
| **`score_agent` tool** | 0–100 trust score + risk signals |
| **`preflight_check` tool** | Go/no-go decision before payment |

**Next steps**: Use `get_trust_receipt` for signed on-chain receipts (x402 paid, $0.01 USDC).

Docs: [intel.twzrd.xyz](https://intel.twzrd.xyz) | PyPI: `pip install twzrd-agent-intel`
